<!-- nav -->
[← 5 · Deciding what you can now](05_Deciding_What_You_Can_Now.ipynb) · [Home](../../README.md) · [7 · One constraint, many languages →](07_One_Constraint_Many_Languages.ipynb)

# 6 · Analyzing and rewriting constraints

**The problem.** The QA team has hundreds of stored constraints. Before a schema change, they need to know which
constraints read the property being renamed. For reviews, they want every constraint written the same way, with the
property first. Both are programs over constraints: one reads them, the other rewrites them.

Because a constraint is data, those programs are ordinary code over a small tree. This case study writes both, and shows
the tools that make them short: `walk`, `fold`, `same`, and rewriting constraints.

In [1]:
from mbse.Expressions import Expressions as E
from mbse.Expressions.Dialects.Python import Text
from mbse.Expressions.Framework import Symbolics, Terms
from mbse.Expressions.Framework.Translators import Hole, Pairwise, Pattern, Rule
from toolkit import show

this, c = E.variable("this"), E.variable("c")
constraints = {
    "in_range": Text.FromFunction(lambda this: 2.0 <= this.celsius <= 8.0).data,
    "on_time": Text.FromFunction(lambda this, to_go: this.hours + to_go <= 72).data,
    "allowed": this.readings.count_where("c", c.gt(8.0)).le(2).and_(this.readings.max().le(12.0)).data,
    "reported": Text.FromFunction(lambda this: this.celsius is not None).data,
}

## Step 1: a constraint is a tree of terms

Every term has a **form**: its `kind` (literal, variable, operation, let, quantifier), its `attributes` (a literal's
value, an operation's name) and its ordered `arguments`, which are terms. `Terms.walk` visits each term once:

In [2]:
for term in Terms.walk(constraints["in_range"]):
    form = term.form()
    print(f"{form.kind:10}", form.attributes, len(form.arguments))

operation  {'name': 'and'} 2
operation  {'name': 'le'} 2
literal    {'value': 2.0} 0
operation  {'name': 'get'} 2
variable   {'name': 'this'} 0
literal    {'value': 'celsius'} 0
operation  {'name': 'le'} 2
literal    {'value': 8.0} 0


`get(this, 'celsius')` appears once in the walk, although the constraint uses it twice: it's one term, shared, as it was
saved once in case study 1.

## Step 2: which constraints read a property

A `get` or `has` whose second argument is a literal names the property it reads. Collecting those is a loop over the
walk:

In [3]:
def reads(constraint):
    found = set()
    for term in Terms.walk(constraint):
        form = term.form()
        if form.kind == "operation" and form.attributes["name"] in ("get", "has"):
            name = form.arguments[1].form()
            if name.kind == "literal":
                found.add(name.attributes["value"])
    return found


{name: sorted(reads(constraint)) for name, constraint in constraints.items()}

{'in_range': ['celsius'],
 'on_time': ['hours'],
 'allowed': ['readings'],
 'reported': ['celsius']}

Renaming `celsius` touches two of the four constraints. `Symbolics.free` answers the other usual question, which
variables a constraint needs, and knows that a quantifier's `c` is bound inside it:

In [4]:
{name: sorted(Symbolics.free(constraint)) for name, constraint in constraints.items()}

{'in_range': ['this'],
 'on_time': ['this', 'to_go'],
 'allowed': ['this'],
 'reported': ['this']}

## Step 3: fold, bottom up

`Terms.fold(constraint, f)` calls `f(term, results)` once per term, children first, where `results` are what `f` gave
for the term's arguments. Anything computed from the parts is a fold. Here's the toolkit's `show`, in short:

In [5]:
def render(term, arguments):
    form = term.form()
    if form.kind == "literal":
        return repr(form.attributes["value"])
    if form.kind == "variable":
        return form.attributes["name"]
    if form.kind == "quantifier":
        return f"{form.attributes['quantifier']}({arguments[0]}, {form.attributes['name']} => {arguments[1]})"
    return f"{form.attributes['name']}({', '.join(arguments)})"


print(Terms.fold(constraints["allowed"], render))
print(Terms.fold(constraints["allowed"], render) == show(constraints["allowed"]))
Terms.fold(constraints["allowed"], lambda term, depths: 1 + max(depths, default=0))  # how deep it nests

and(le(count(get(this, 'readings'), c => gt(c, 8.0)), 2), le(max(get(this, 'readings')), 12.0))
True


5

The toolkit's version adds lets and domains. Because each term is folded once, a shared term costs nothing extra.

## Step 4: same

`Terms.same(a, b)` compares two constraints by structure: the same kinds, attributes and arguments, all the way down.
Two constraints written separately are the same when they say the same thing the same way:

In [6]:
again = Text.FromFunction(lambda this: 2.0 <= this.celsius and this.celsius <= 8.0).data
swapped = Text.FromFunction(lambda this: this.celsius >= 2.0 and this.celsius <= 8.0).data
Terms.same(constraints["in_range"], again), Terms.same(constraints["in_range"], swapped)

(True, False)

## Step 5: rewriting

`2.0 <= this.celsius` and `this.celsius >= 2.0` mean the same, and reviews would be easier if every constraint used the
second. A **rewriting** is a list of rules, each pairing a pattern with its replacement. Patterns have **holes**,
which match any term (`A`) or any attribute value (`K`, `N`), and a rule is applied wherever its pattern matches. The
most specific pattern wins, so the general rules say what stays as it is, and the specific one what changes:

In [7]:
A, B, K, N = Hole("A"), Hole("B"), Hole("K"), Hole("N", str)
keep = [Rule(Pattern("variable", name=N), Pattern("variable", name=N), "forward"),
        Rule(Pattern("literal", value=K), Pattern("literal", value=K), "forward"),
        Rule(Pattern("operation", A, name=N), Pattern("operation", A, name=N), "forward"),
        Rule(Pattern("operation", A, B, name=N), Pattern("operation", A, B, name=N), "forward")]
flip = [Rule(Pattern("operation", Pattern("literal", value=K), A, name=op),
             Pattern("operation", A, Pattern("literal", value=K), name=mirror), "forward")
        for op, mirror in [("le", "ge"), ("lt", "gt"), ("ge", "le"), ("gt", "lt")]]
property_first = Pairwise(E.DIALECT, E.DIALECT, keep + flip)
rewritten = property_first.forward(constraints["in_range"])
print(show(rewritten))
Terms.same(rewritten, swapped)

and(ge(get(this, 'celsius'), 2.0), le(get(this, 'celsius'), 8.0))


True

`forward` makes a new constraint and leaves the old one as it was: expressions are never changed in place. The rewritten
constraint is now the same as the one written property first.

The rewriting above handles terms with one or two arguments; `allowed` has a quantifier, which it has no rule for, and
it says so rather than guessing:

In [8]:
try:
    property_first.forward(constraints["allowed"])
except ValueError as error:
    print(error)

Basic quantifier 'count' has no Basic counterpart


## Why it works this way

- **A small tree, the same in every dialect.** Five kinds of term, each with a form, so an analysis is a loop or a fold,
  and it works for any dialect's expressions (case study 7), which have forms too.
- **Rewriting is translation.** A rewriting is a translator from a dialect to itself, written with the same rules as
  the translators between languages, which case study 7 uses.
- **Refuse what isn't covered.** A rewriting that met a term it had no rule for and copied it silently could change
  what a constraint means without anyone noticing.

<!-- nav -->
[← 5 · Deciding what you can now](05_Deciding_What_You_Can_Now.ipynb) · [Home](../../README.md) · [7 · One constraint, many languages →](07_One_Constraint_Many_Languages.ipynb)